# Hyperedge prediction — contact-primary (HyperLap)

Loads **`../www21-hyperlap-master/dataset/contact-primary_graph.txt`**, builds **10** train/test splits **once** (reused for all embedding runs). For each split, runs **HMRW** then **Hyper2Vec** for each **(p, q)** in `[[1, 1], [2, 0.5], [0.5, 2]]`.

Scores held-out positives vs negatives with `evaluate`. Results in a **DataFrame**; each embedding step is **printed**.

In [ ]:
from pathlib import Path

from hypergraphx import Hypergraph

txt_path = Path("..") / "www21-hyperlap-master" / "dataset" / "contact-primary_graph.txt"

edge_list = []
with open(txt_path, "r") as f:
    for line in f:
        line = line.strip()
        if line == "":
            continue
        tokens = line.replace(",", " ").split()
        edge = [int(x) for x in tokens]
        edge_list.append(edge)

H_full = Hypergraph(edge_list)

print(f"Loaded: {txt_path.resolve()}")
print(f"|V|={len(H_full.get_nodes())}, |E|={len(H_full.get_edges())}")

In [ ]:
import matplotlib.pyplot as plt

# Histogram of node degrees from H_full
# degree_distribution() is expected to return a mapping degree -> count.
deg_dist = H_full.degree_distribution()

degrees = list(deg_dist.keys())
counts = list(deg_dist.values())

plt.figure(figsize=(8, 4))
plt.bar(degrees, counts, width=0.9, edgecolor="black")
plt.xlabel("Node degree")
plt.ylabel("Number of nodes")
plt.title("Degree distribution of contact-primary hypergraph")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Total nodes in distribution: {sum(counts)}")

In [ ]:
import matplotlib.pyplot as plt

# Histogram of hyperedge sizes
sizes = list(H_full.get_sizes())

plt.figure(figsize=(8, 4))
plt.hist(sizes, bins=30, edgecolor="black")
plt.xlabel("Hyperedge size")
plt.ylabel("Frequency")
plt.title("Hyperedge size distribution of contact-primary hypergraph")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Number of hyperedges: {len(sizes)}")
print(f"Min size: {min(sizes)}, Max size: {max(sizes)}")

In [ ]:
H_full.get_sizes()

In [ ]:
import pandas as pd

from src.hypergraph_train_test_split import (
    split_hypergraph_train_test,
    generate_negative_hyperedges,
    evaluate,
)
from src.utils import hmrw_embeddings, hyper2vec_embeddings

# --- Parameters (edit here) ---
test_ratio = 0.1
n_splits = 20
split_seeds = list(range(n_splits))

p_q_pairs = [
    [1.0, 1.0],
    [2.0, 0.5],
    [0.5, 2.0],
    [4.0, 1],
    [1, 4.0],   
]

r_hmrw = 1.0
gamma = -1.0
r_h2v = 0.0

num_walks = 5
walk_length = 100
dim = 128

print("Setup:")
print(f"  test_ratio={test_ratio}, split_seeds={split_seeds}")
print(f"  p_q_pairs={p_q_pairs}")
print(f"  HMRW: r={r_hmrw}, gamma={gamma}; Hyper2Vec: r={r_h2v}")
print(f"  walks: num_walks={num_walks}, walk_length={walk_length}, dim={dim}")

In [ ]:
from __future__ import annotations

import warnings

import numpy as np
from gensim.models import Word2Vec
from hypergraphx import Hypergraph
from hypergraphx.representations.projections import clique_projection


def hmrw_exp_embeddings(
    hg: Hypergraph,
    *,
    p: float = 1.0,
    q: float = 1.0,
    r: float = 1.0,
    gamma: float = -1.0,
    num_walks: int = 10,
    walk_length: int = 80,
    window: int = 10,
    dim: int = 128,
    seed: int | None = None,
    workers: int = 1,
):
    """
    Hyperedge-Memory Random Walk (HMRW), then Skip-gram.

    Based on your method in `hyperedge_aware_rw.pdf`:
    - Keep state (u, v, e): previous node u, current node v, previous hyperedge e.
    - Stage 1: sample next hyperedge e' incident to v with
      pi(e'|e,u,v) ∝ a_edge(e',e,u) * h(v,e') / |e'|.
    - Stage 2: sample next node x in e' with
      pi(x|e',v,e,u) ∝ a_node(x,e,u) * h(x,e') / d(x).
    ``num_walks`` is the number of walks **per node**.
    """
    nodes = sorted(hg.get_nodes())
    n = len(nodes)
    if n == 0:
        return np.zeros((0, dim), dtype=float)

    edges = [frozenset(e) for e in hg.get_edges()]
    m = len(edges)
    if m == 0:
        return np.zeros((n, dim), dtype=float)

    idx = {u: i for i, u in enumerate(nodes)}

    # Node degree d(x) = number of incident hyperedges.
    d = np.zeros(n, dtype=float)
    inc_edges = [[] for _ in range(n)]  # incident edge ids for each node index
    for e_i, e in enumerate(edges):
        for u in e:
            i = idx[u]
            d[i] += 1.0
            inc_edges[i].append(e_i)

    rng = np.random.default_rng(seed)
    walks: list[list[int]] = []
    isolated_step_count = 0

    for _ in range(num_walks):
        for start in rng.permutation(n):
            v_i = int(start)
            walk = [v_i]
            prev_u_i: int | None = None
            prev_e_i: int | None = None

            for _ in range(walk_length - 1):
                # --- Stage 1: choose next hyperedge e' incident to current node v.
                cand_e = inc_edges[v_i]
                if not cand_e:
                    isolated_step_count += 1
                    walk.append(v_i)
                    continue

                raw_e = np.zeros(len(cand_e), dtype=float)
                for k, e2_i in enumerate(cand_e):
                    e2 = edges[e2_i]
                    if len(e2) == 0:
                        continue
                    if prev_e_i is None or prev_u_i is None:
                        a_edge = 1.0
                    elif e2_i == prev_e_i:
                        a_edge = 1.0 / p
                    elif len((edges[prev_e_i] - {nodes[v_i]}) & (e2 - {nodes[v_i]})) > 0:
                        a_edge = 1.0
                    else:
                        a_edge = 1.0 / q
                    size_term = max(len(e2) - 1, 1)
                    raw_e[k] = a_edge * (size_term ** gamma)  # h(v,e')=1 since e' incident to v

                s_e = raw_e.sum()
                if s_e <= 0 or not np.isfinite(s_e):
                    next_e_i = int(rng.choice(cand_e))
                else:
                    next_e_i = cand_e[int(rng.choice(len(cand_e), p=raw_e / s_e))]

                # --- Stage 2: choose next node x in e'.
                e2 = edges[next_e_i]
                # cand_x = [idx[x] for x in e2]
                cand_x= [idx[x] for x in e2 if idx[x] != v_i]
                raw_x = np.zeros(len(cand_x), dtype=float)
                prev_e_nodes = edges[prev_e_i] if prev_e_i is not None else None

                for k, x_i in enumerate(cand_x):
                    if d[x_i] <= 0:
                        continue
                    if prev_u_i is None:
                        a_node = 1.0
                    elif prev_e_nodes is not None and nodes[x_i] in prev_e_nodes:
                        a_node = 1.0
                    else:
                        a_node = 1.0 / r
                    raw_x[k] = a_node 

                s_x = raw_x.sum()
                x_i = cand_x[int(rng.choice(len(cand_x), p=raw_x / s_x))]

                # Advance walk memory: (u,v,e) <- (v,x,e').
                prev_u_i = v_i
                prev_e_i = next_e_i
                v_i = x_i
                walk.append(v_i)

            walks.append(walk)

    if isolated_step_count > 0:
        warnings.warn(
            f"HMRW encountered {isolated_step_count} step(s) at nodes with no incident hyperedges; "
            "the walk stayed on the same node for those steps.",
            RuntimeWarning,
            stacklevel=2,
        )

    w2v = Word2Vec(
        sentences=[[str(i) for i in w] for w in walks],
        vector_size=dim,
        window=window,
        min_count=1,
        sg=1,
        workers=workers,
        seed=seed if seed is not None else 0,
        epochs=20,
    )

    out = np.zeros((n, dim), dtype=float)
    for i in range(n):
        k = str(i)
        if k in w2v.wv:
            out[i] = w2v.wv[k]
    return out



## Train / test splits (run once, then reuse)

Each entry has `H_train`, `test_edges`, `negative_edges`, `nodes_train`, `fold`, `split_seed`.

In [ ]:
import pickle

splits = []
split_rows_to_save = []

for fold, split_seed in enumerate(split_seeds):
    split_seed += 42
    H_train, test_edges = split_hypergraph_train_test(
        H_full, test_ratio=test_ratio, seed=split_seed
    )
    negative_edges = generate_negative_hyperedges(
        H_full, test_edges, seed=split_seed
    )
    nodes_train = sorted(H_train.get_nodes())

    sp = {
        "fold": fold,
        "split_seed": split_seed,
        "H_train": H_train,
        "test_edges": test_edges,
        "negative_edges": negative_edges,
        "nodes_train": nodes_train,
    }
    splits.append(sp)

    split_rows_to_save.append(
        {
            "fold": fold,
            "split_seed": split_seed,
            "train_edge_list": list(H_train.get_edges()),
            "test_edges": test_edges,
            "negative_edges": negative_edges,
            "nodes_train": nodes_train,
        }
    )

    print(
        f"Fold {fold} (split_seed={split_seed}): "
        f"train |E|={len(H_train.get_edges())}, "
        f"pos={len(test_edges)}, neg={len(negative_edges)}"
    )

splits_out_path = Path("results") / "contact_primary" / "hedge_prediction_overlap_10perc_20splits.pkl"
splits_out_path.parent.mkdir(parents=True, exist_ok=True)
with open(splits_out_path, "wb") as f:
    pickle.dump(
        {
            "dataset_path": str(txt_path),
            "test_ratio": test_ratio,
            "split_seeds": split_seeds,
            "splits": split_rows_to_save,
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

print(f"\nCached {len(splits)} splits.")
print(f"Saved splits: {splits_out_path}")

## Embeddings + evaluation (reuses `splits`)

In [ ]:
num_walks = 10
walk_length = 40
dim = 128

In [ ]:
rows = []

for p, q in p_q_pairs:
    print(f"\n=== (p, q)=({p}, {q}) ===")

    for sp in splits:
        fold = sp["fold"]
        split_seed = sp["split_seed"]
        H_train = sp["H_train"]
        test_edges = sp["test_edges"]
        negative_edges = sp["negative_edges"]
        nodes_train = sp["nodes_train"]
        emb_seed = split_seed

        print(
            f"  Fold {fold} (split_seed={split_seed}) | "
            f"train |E|={len(H_train.get_edges())}, "
            f"pos={len(test_edges)}, neg={len(negative_edges)}"
        )

        X = hmrw_embeddings(
            H_train,
            p=p,
            q=q,
            r=r_hmrw,
            gamma=gamma,
            num_walks=num_walks,
            walk_length=walk_length,
            dim=dim,
            seed=emb_seed,
        )
        emb = {node: X[i] for i, node in enumerate(nodes_train)}
        auc_h, ap_h = evaluate(test_edges, negative_edges, emb)
        row_h = {
            "fold": fold,
            "split_seed": split_seed,
            "method": "hmrw",
            "p": p,
            "q": q,
            "r": r_hmrw,
            "gamma": gamma,
            "auc": auc_h,
            "ap": ap_h,
            "n_train_edges": len(H_train.get_edges()),
            "n_test_pos": len(test_edges),
            "n_test_neg": len(negative_edges),
        }
        rows.append(row_h)
        print(
            f"    [HMRW] AUC={auc_h:.4f}, AP={ap_h:.4f}  "
            f"(rows={len(rows)})"
        )
        
        X = hmrw_exp_embeddings(
            H_train,
            p=p,
            q=q,
            r=r_hmrw,
            gamma=gamma,
            num_walks=num_walks,
            walk_length=walk_length,
            dim=dim,
            seed=emb_seed,
        )
        emb = {node: X[i] for i, node in enumerate(nodes_train)}
        auc_h, ap_h = evaluate(test_edges, negative_edges, emb)
        row_h = {
            "fold": fold,
            "split_seed": split_seed,
            "method": "hmrw exp",
            "p": p,
            "q": q,
            "r": r_hmrw,
            "gamma": gamma,
            "auc": auc_h,
            "ap": ap_h,
            "n_train_edges": len(H_train.get_edges()),
            "n_test_pos": len(test_edges),
            "n_test_neg": len(negative_edges),
        }
        rows.append(row_h)
        print(
            f"    [HMRW exp] AUC={auc_h:.4f}, AP={ap_h:.4f}  "
            f"(rows={len(rows)})"
        )

        X2 = hyper2vec_embeddings(
            H_train,
            p=p,
            q=q,
            r=r_h2v,
            num_walks=num_walks,
            walk_length=walk_length,
            dim=dim,
            seed=emb_seed,
        )
        emb2 = {node: X2[i] for i, node in enumerate(nodes_train)}
        auc_v, ap_v = evaluate(test_edges, negative_edges, emb2)
        row_v = {
            "fold": fold,
            "split_seed": split_seed,
            "method": "hyper2vec",
            "p": p,
            "q": q,
            "r": r_h2v,
            "gamma": None,
            "auc": auc_v,
            "ap": ap_v,
            "n_train_edges": len(H_train.get_edges()),
            "n_test_pos": len(test_edges),
            "n_test_neg": len(negative_edges),
        }
        rows.append(row_v)
        print(
            f"    [Hyper2Vec] AUC={auc_v:.4f}, AP={ap_v:.4f}  "
            f"(rows={len(rows)})"
        )

results_df = pd.DataFrame(rows)
print(f"\nDone. DataFrame shape: {results_df.shape}")
results_df.head(12)

In [ ]:
results_df = pd.DataFrame(rows)

# Mean AUC / AP by method and (p, q) over folds
summary = (
    results_df.groupby(["method", "p", "q", "r"], as_index=False)[["auc", "ap"]]
    .agg(["mean", "std"])
    .round(4)
)
summary

In [ ]:
from run_hedge_prediction_experiments import run_hedge_prediction

csv_path = run_hedge_prediction(
    "contact_primary",
    "../www21-hyperlap-master/dataset/contact-primary_graph.txt",
)
print("Results CSV:", csv_path.resolve())